# Day 077 | Physics-Informed Neural Networks (PINNs)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning

**Author:** Ripon Chandra Malo

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Physics-informed loss | Embed PDE constraints into the loss function | Terzaghi consolidation equation as a regularizer |
| Collocation points | Points where PDE residual is evaluated | Grid of (z, t) points across the soil layer |
| Boundary conditions | Enforce physical constraints at boundaries | Zero excess PWP at drainage boundary |
| Initial conditions | Enforce known state at t=0 | Uniform initial excess pore pressure |
| PINN vs data-driven | Compare physics-constrained vs pure data models | Better extrapolation with fewer data points |

---

## Our Geotechnical Problem

We solve the **1D Terzaghi consolidation equation** using a neural network.
This PDE governs how excess pore water pressure dissipates in a saturated clay layer:

$$\frac{\partial u}{\partial t} = c_v \frac{\partial^2 u}{\partial z^2}$$

where:
- $u(z, t)$ = excess pore water pressure
- $c_v$ = coefficient of consolidation
- $z$ = depth, $t$ = time

**Boundary conditions:**
- $u(0, t) = 0$ (free-draining top surface)
- $\partial u / \partial z (H, t) = 0$ (impermeable bottom)
- $u(z, 0) = u_0$ (uniform initial excess PWP)

---

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

# Use double precision for PDE derivatives
torch.set_default_dtype(torch.float64)

print(f"PyTorch version: {torch.__version__}")

## Part 1: Analytical Solution (Ground Truth)

The analytical solution for 1D Terzaghi consolidation with single drainage is:

$$u(z, t) = \sum_{m=0}^{\infty} \frac{2 u_0}{M} \sin\left(\frac{M z}{H}\right) \exp\left(-M^2 T_v\right)$$

where $M = \frac{\pi}{2}(2m+1)$ and $T_v = \frac{c_v t}{H^2}$ is the time factor.

We use this to validate our PINN.

---

In [ ]:
# ───────────────────────────────────────────────────────
# PROBLEM PARAMETERS
# ───────────────────────────────────────────────────────

H = 10.0        # clay layer thickness (m)
cv = 2.0        # coefficient of consolidation (m^2/year)
u0 = 100.0      # initial excess pore pressure (kPa)
T_max = 10.0    # simulation time (years)

print(f"Clay layer:  H = {H} m")
print(f"cv = {cv} m^2/year")
print(f"Initial PWP: u0 = {u0} kPa")
print(f"Time range:  0 to {T_max} years")
print(f"\nTime factor at T_max: Tv = {cv * T_max / H**2:.3f}")

In [ ]:
# ───────────────────────────────────────────────────────
# ANALYTICAL SOLUTION
# ───────────────────────────────────────────────────────

def terzaghi_analytical(z, t, H=10.0, cv=2.0, u0=100.0, n_terms=100):
    """
    Analytical solution for 1D Terzaghi consolidation.
    Single drainage (top free, bottom impermeable).
    
    Parameters:
    -----------
    z : depth (0 = top, H = bottom)
    t : time
    """
    u = np.zeros_like(z, dtype=float)
    
    for m in range(n_terms):
        M = np.pi / 2 * (2 * m + 1)
        Tv = cv * t / H**2
        u += (2 * u0 / M) * np.sin(M * z / H) * np.exp(-M**2 * Tv)
    
    return u

# Verify: plot at different times
z_plot = np.linspace(0, H, 100)
times = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

fig, ax = plt.subplots(figsize=(8, 7))
for t in times:
    u = terzaghi_analytical(z_plot, t)
    ax.plot(u, z_plot, linewidth=2, label=f't = {t} yr')

ax.set_xlabel('Excess Pore Pressure (kPa)')
ax.set_ylabel('Depth (m)')
ax.set_title('Terzaghi Consolidation  |  Analytical Solution')
ax.invert_yaxis()
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, u0 * 1.05)
plt.tight_layout()
plt.show()

## Part 2: Building the PINN

The PINN approximates $u(z, t)$ with a neural network $u_{\theta}(z, t)$.

The loss function has **four components**:

$$\mathcal{L} = \lambda_{\text{pde}} \mathcal{L}_{\text{pde}} + \lambda_{\text{ic}} \mathcal{L}_{\text{ic}} + \lambda_{\text{bc}} \mathcal{L}_{\text{bc}} + \lambda_{\text{data}} \mathcal{L}_{\text{data}}$$

| Loss Component | What It Enforces |
|---------------|------------------|
| $\mathcal{L}_{\text{pde}}$ | PDE residual: $u_t - c_v u_{zz} = 0$ at collocation points |
| $\mathcal{L}_{\text{ic}}$ | Initial condition: $u(z, 0) = u_0$ |
| $\mathcal{L}_{\text{bc}}$ | Boundary conditions: $u(0, t) = 0$, $u_z(H, t) = 0$ |
| $\mathcal{L}_{\text{data}}$ | Fit to any available measurement data |

---

In [ ]:
# ───────────────────────────────────────────────────────
# PINN NETWORK ARCHITECTURE
# ───────────────────────────────────────────────────────

class PINN(nn.Module):
    """
    Physics-Informed Neural Network for 1D consolidation.
    
    Input:  (z, t) — depth and time
    Output: u      — excess pore water pressure
    """
    
    def __init__(self, hidden_layers=4, hidden_size=40):
        super().__init__()
        
        layers = []
        layers.append(nn.Linear(2, hidden_size))  # input: (z, t)
        layers.append(nn.Tanh())
        
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_size, 1))  # output: u
        
        self.net = nn.Sequential(*layers)
        
        # Initialize weights (Xavier)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, z, t):
        inputs = torch.cat([z, t], dim=1)
        return self.net(inputs)

model = PINN(hidden_layers=4, hidden_size=40)
n_params = sum(p.numel() for p in model.parameters())
print(f"PINN parameters: {n_params:,}")
print(f"\nArchitecture:\n{model}")

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE COLLOCATION AND BOUNDARY POINTS
# ───────────────────────────────────────────────────────

def generate_training_points(n_interior=2000, n_bc=200, n_ic=200, n_data=20):
    """
    Generate points for PINN training.
    
    Returns dictionaries of (z, t) points for each loss component.
    """
    # Interior collocation points (random in domain)
    z_int = torch.rand(n_interior, 1) * H
    t_int = torch.rand(n_interior, 1) * T_max
    
    # Boundary condition: u(0, t) = 0 (top, free drainage)
    z_bc_top = torch.zeros(n_bc, 1)
    t_bc_top = torch.rand(n_bc, 1) * T_max
    
    # Boundary condition: du/dz(H, t) = 0 (bottom, impermeable)
    z_bc_bot = torch.full((n_bc, 1), H)
    t_bc_bot = torch.rand(n_bc, 1) * T_max
    
    # Initial condition: u(z, 0) = u0
    z_ic = torch.rand(n_ic, 1) * H
    t_ic = torch.zeros(n_ic, 1)
    u_ic = torch.full((n_ic, 1), u0)
    
    # Sparse measurement data (simulated from analytical solution)
    z_data = torch.rand(n_data, 1) * H
    t_data = torch.rand(n_data, 1) * T_max
    u_data = torch.tensor(
        terzaghi_analytical(z_data.numpy(), t_data.numpy()),
        dtype=torch.float64
    )
    # Add measurement noise
    u_data += torch.randn_like(u_data) * 2.0
    
    return {
        'interior': (z_int, t_int),
        'bc_top':   (z_bc_top, t_bc_top),
        'bc_bot':   (z_bc_bot, t_bc_bot),
        'ic':       (z_ic, t_ic, u_ic),
        'data':     (z_data, t_data, u_data),
    }

points = generate_training_points()
print("Training points generated:")
print(f"  Interior (PDE):      {points['interior'][0].shape[0]}")
print(f"  BC top (u=0):        {points['bc_top'][0].shape[0]}")
print(f"  BC bottom (du/dz=0): {points['bc_bot'][0].shape[0]}")
print(f"  Initial condition:   {points['ic'][0].shape[0]}")
print(f"  Measurement data:    {points['data'][0].shape[0]}")

## Part 3: Computing PDE Residual with Automatic Differentiation

The key insight of PINNs: we use PyTorch's `autograd` to compute derivatives of the
network output with respect to its inputs:

$$\text{Residual} = \frac{\partial u_{\theta}}{\partial t} - c_v \frac{\partial^2 u_{\theta}}{\partial z^2}$$

If the network satisfies the PDE, the residual is zero everywhere.

---

In [ ]:
# ───────────────────────────────────────────────────────
# PDE RESIDUAL COMPUTATION
# ───────────────────────────────────────────────────────

def compute_pde_residual(model, z, t, cv=2.0):
    """
    Compute the Terzaghi PDE residual:
        R = du/dt - cv * d^2u/dz^2
    
    Uses automatic differentiation for derivatives.
    """
    z.requires_grad_(True)
    t.requires_grad_(True)
    
    u = model(z, t)
    
    # First derivatives
    du_dt = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    du_dz = torch.autograd.grad(u, z, grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    
    # Second derivative
    d2u_dz2 = torch.autograd.grad(du_dz, z, grad_outputs=torch.ones_like(du_dz),
                                  create_graph=True)[0]
    
    # PDE residual
    residual = du_dt - cv * d2u_dz2
    
    return residual, du_dz

# Test: residual before training (should be non-zero)
z_test = torch.rand(5, 1) * H
t_test = torch.rand(5, 1) * T_max
res_test, _ = compute_pde_residual(model, z_test, t_test)
print(f"PDE residual before training (should be non-zero):")
print(f"  {res_test.detach().squeeze().numpy().round(3)}")

In [ ]:
# ───────────────────────────────────────────────────────
# PINN LOSS FUNCTION
# ───────────────────────────────────────────────────────

def pinn_loss(model, points, cv=2.0, weights=None):
    """
    Compute total PINN loss with four components.
    
    weights: dict with keys 'pde', 'bc', 'ic', 'data'
    """
    if weights is None:
        weights = {'pde': 1.0, 'bc': 10.0, 'ic': 10.0, 'data': 1.0}
    
    # 1. PDE residual loss
    z_int, t_int = points['interior']
    residual, _ = compute_pde_residual(model, z_int, t_int, cv)
    loss_pde = (residual ** 2).mean()
    
    # 2. Top boundary condition: u(0, t) = 0
    z_top, t_top = points['bc_top']
    u_top = model(z_top, t_top)
    loss_bc_top = (u_top ** 2).mean()
    
    # 3. Bottom boundary condition: du/dz(H, t) = 0
    z_bot, t_bot = points['bc_bot']
    z_bot_req = z_bot.clone().requires_grad_(True)
    u_bot = model(z_bot_req, t_bot)
    du_dz_bot = torch.autograd.grad(u_bot, z_bot_req,
                                     grad_outputs=torch.ones_like(u_bot),
                                     create_graph=True)[0]
    loss_bc_bot = (du_dz_bot ** 2).mean()
    
    loss_bc = loss_bc_top + loss_bc_bot
    
    # 4. Initial condition: u(z, 0) = u0
    z_ic, t_ic, u_ic_true = points['ic']
    u_ic_pred = model(z_ic, t_ic)
    loss_ic = ((u_ic_pred - u_ic_true) ** 2).mean()
    
    # 5. Data loss (sparse measurements)
    z_d, t_d, u_d = points['data']
    u_d_pred = model(z_d, t_d)
    loss_data = ((u_d_pred - u_d) ** 2).mean()
    
    # Total weighted loss
    total = (weights['pde'] * loss_pde + weights['bc'] * loss_bc +
             weights['ic'] * loss_ic + weights['data'] * loss_data)
    
    return total, {
        'pde': loss_pde.item(),
        'bc': loss_bc.item(),
        'ic': loss_ic.item(),
        'data': loss_data.item(),
        'total': total.item()
    }

In [ ]:
# ───────────────────────────────────────────────────────
# TRAIN THE PINN
# ───────────────────────────────────────────────────────

torch.manual_seed(42)
model = PINN(hidden_layers=4, hidden_size=40)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=700, gamma=0.5)

epochs = 2000
history = {'total': [], 'pde': [], 'bc': [], 'ic': [], 'data': []}

loss_weights = {'pde': 1.0, 'bc': 10.0, 'ic': 10.0, 'data': 1.0}

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # Resample collocation points every 500 epochs for better coverage
    if epoch % 500 == 0:
        points = generate_training_points(n_interior=1000, n_bc=200, n_ic=200, n_data=20)
    
    loss, loss_dict = pinn_loss(model, points, cv=cv, weights=loss_weights)
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    for key in history:
        history[key].append(loss_dict[key])
    
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1:5d}/{epochs} | Total: {loss_dict['total']:.4e}  "
              f"PDE: {loss_dict['pde']:.4e}  BC: {loss_dict['bc']:.4e}  "
              f"IC: {loss_dict['ic']:.4e}  Data: {loss_dict['data']:.4e}")

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING CURVES BY LOSS COMPONENT
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(history['total'], 'k-', linewidth=2, label='Total')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Total PINN Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].semilogy(history['pde'], label='PDE residual', linewidth=1.5)
axes[1].semilogy(history['bc'], label='Boundary conditions', linewidth=1.5)
axes[1].semilogy(history['ic'], label='Initial condition', linewidth=1.5)
axes[1].semilogy(history['data'], label='Data fitting', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Individual Loss Components')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.suptitle('PINN Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Comparing PINN vs Analytical Solution

Now let us see how well the PINN has learned the Terzaghi consolidation solution.
We compare against the analytical solution at multiple time snapshots.

---

In [ ]:
# ───────────────────────────────────────────────────────
# PINN PREDICTIONS vs ANALYTICAL SOLUTION
# ───────────────────────────────────────────────────────

model.eval()

z_eval = np.linspace(0, H, 100)
times_eval = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, t_val in zip(axes, times_eval):
    # Analytical
    u_analytical = terzaghi_analytical(z_eval, t_val)
    
    # PINN prediction
    z_t = torch.tensor(z_eval.reshape(-1, 1), dtype=torch.float64)
    t_t = torch.full_like(z_t, t_val)
    with torch.no_grad():
        u_pinn = model(z_t, t_t).numpy().flatten()
    
    ax.plot(u_analytical, z_eval, 'b-', linewidth=2.5, label='Analytical')
    ax.plot(u_pinn, z_eval, 'r--', linewidth=2, label='PINN')
    
    rmse = np.sqrt(((u_analytical - u_pinn)**2).mean())
    ax.set_title(f't = {t_val} yr  |  RMSE = {rmse:.2f} kPa')
    ax.set_xlabel('u (kPa)')
    ax.set_ylabel('Depth (m)')
    ax.invert_yaxis()
    ax.set_xlim(-5, 105)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('PINN vs Analytical  |  Terzaghi Consolidation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# 2D CONTOUR PLOT: u(z, t)
# ───────────────────────────────────────────────────────

z_grid = np.linspace(0, H, 80)
t_grid = np.linspace(0.05, T_max, 80)
Z_mesh, T_mesh = np.meshgrid(z_grid, t_grid)

# Analytical
U_analytical = np.zeros_like(Z_mesh)
for i in range(Z_mesh.shape[0]):
    for j in range(Z_mesh.shape[1]):
        U_analytical[i, j] = terzaghi_analytical(
            np.array([Z_mesh[i, j]]), T_mesh[i, j]
        )

# PINN
z_flat = torch.tensor(Z_mesh.flatten().reshape(-1, 1), dtype=torch.float64)
t_flat = torch.tensor(T_mesh.flatten().reshape(-1, 1), dtype=torch.float64)
with torch.no_grad():
    U_pinn = model(z_flat, t_flat).numpy().reshape(Z_mesh.shape)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].contourf(T_mesh, Z_mesh, U_analytical, levels=20, cmap='hot_r')
axes[0].set_title('Analytical Solution')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Depth (m)')
axes[0].invert_yaxis()
plt.colorbar(im0, ax=axes[0], label='u (kPa)')

im1 = axes[1].contourf(T_mesh, Z_mesh, U_pinn, levels=20, cmap='hot_r')
axes[1].set_title('PINN Prediction')
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Depth (m)')
axes[1].invert_yaxis()
plt.colorbar(im1, ax=axes[1], label='u (kPa)')

error = np.abs(U_analytical - U_pinn)
im2 = axes[2].contourf(T_mesh, Z_mesh, error, levels=20, cmap='Reds')
axes[2].set_title(f'Absolute Error (max={error.max():.2f} kPa)')
axes[2].set_xlabel('Time (years)')
axes[2].set_ylabel('Depth (m)')
axes[2].invert_yaxis()
plt.colorbar(im2, ax=axes[2], label='|error| (kPa)')

plt.tight_layout()
plt.show()

## Part 5: PINN vs Pure Data-Driven Model

We now compare the PINN against a **pure data-driven network** (same architecture,
no physics in the loss). The data-driven model is trained only on sparse measurements.

This demonstrates the key advantage of PINNs: **they extrapolate better** because
the physics constrains the solution space.

---

In [ ]:
# ───────────────────────────────────────────────────────
# PURE DATA-DRIVEN MODEL (NO PHYSICS)
# ───────────────────────────────────────────────────────

# Generate more data points for fair comparison
n_data_points = 100
np.random.seed(42)
z_data_dd = np.random.rand(n_data_points) * H
t_data_dd = np.random.rand(n_data_points) * T_max
u_data_dd = terzaghi_analytical(z_data_dd, t_data_dd) + np.random.randn(n_data_points) * 2.0

z_data_t = torch.tensor(z_data_dd.reshape(-1, 1), dtype=torch.float64)
t_data_t = torch.tensor(t_data_dd.reshape(-1, 1), dtype=torch.float64)
u_data_t = torch.tensor(u_data_dd.reshape(-1, 1), dtype=torch.float64)

# Train pure data-driven model
torch.manual_seed(42)
model_dd = PINN(hidden_layers=4, hidden_size=40)
opt_dd = torch.optim.Adam(model_dd.parameters(), lr=1e-3)

dd_losses = []
for epoch in range(2000):
    model_dd.train()
    opt_dd.zero_grad()
    
    u_pred = model_dd(z_data_t, t_data_t)
    loss = ((u_pred - u_data_t) ** 2).mean()
    loss.backward()
    opt_dd.step()
    dd_losses.append(loss.item())
    
    if (epoch + 1) % 2000 == 0:
        print(f"Data-driven Epoch {epoch+1}/2000 | MSE: {loss.item():.4e}")

print(f"Final data-driven MSE: {dd_losses[-1]:.4e}")

In [ ]:
# ───────────────────────────────────────────────────────
# COMPARE PINN vs DATA-DRIVEN
# ───────────────────────────────────────────────────────

model.eval()
model_dd.eval()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

rmse_pinn_list = []
rmse_dd_list = []

for ax, t_val in zip(axes, times_eval):
    # Analytical
    u_analytical = terzaghi_analytical(z_eval, t_val)
    
    # PINN
    z_t = torch.tensor(z_eval.reshape(-1, 1), dtype=torch.float64)
    t_t = torch.full_like(z_t, t_val)
    with torch.no_grad():
        u_pinn = model(z_t, t_t).numpy().flatten()
        u_dd = model_dd(z_t, t_t).numpy().flatten()
    
    rmse_p = np.sqrt(((u_analytical - u_pinn)**2).mean())
    rmse_d = np.sqrt(((u_analytical - u_dd)**2).mean())
    rmse_pinn_list.append(rmse_p)
    rmse_dd_list.append(rmse_d)
    
    ax.plot(u_analytical, z_eval, 'b-', linewidth=2.5, label='Analytical')
    ax.plot(u_pinn, z_eval, 'r--', linewidth=2, label=f'PINN ({rmse_p:.1f})')
    ax.plot(u_dd, z_eval, 'g:', linewidth=2, label=f'Data-only ({rmse_d:.1f})')
    
    ax.set_title(f't = {t_val} yr')
    ax.set_xlabel('u (kPa)')
    ax.set_ylabel('Depth (m)')
    ax.invert_yaxis()
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('PINN vs Data-Driven  |  Terzaghi Consolidation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ───────────────────────────────────────────────────────
# RMSE COMPARISON ACROSS TIME
# ───────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(times_eval))
width = 0.35

bars1 = ax.bar(x_pos - width/2, rmse_pinn_list, width, label='PINN', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, rmse_dd_list, width, label='Data-driven', color='#27ae60', alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels([f't={t}' for t in times_eval])
ax.set_xlabel('Time (years)')
ax.set_ylabel('RMSE (kPa)')
ax.set_title('RMSE Comparison: PINN vs Data-Driven at Different Times')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nOverall RMSE — PINN: {np.mean(rmse_pinn_list):.2f} kPa,  Data-driven: {np.mean(rmse_dd_list):.2f} kPa")

In [ ]:
# ───────────────────────────────────────────────────────
# DEGREE OF CONSOLIDATION FROM PINN
# ───────────────────────────────────────────────────────

def compute_avg_consolidation(model, z_points, t_val):
    """Compute average degree of consolidation U at time t."""
    z_t = torch.tensor(z_points.reshape(-1, 1), dtype=torch.float64)
    t_t = torch.full_like(z_t, t_val)
    with torch.no_grad():
        u_pred = model(z_t, t_t).numpy().flatten()
    # U = 1 - (average u / u0)
    U = 1.0 - np.trapz(u_pred, z_points) / (u0 * H)
    return max(0, min(1, U))

t_range = np.linspace(0.05, T_max, 50)
z_fine = np.linspace(0, H, 200)

U_pinn = [compute_avg_consolidation(model, z_fine, t) for t in t_range]

# Analytical degree of consolidation
U_analytical_list = []
for t in t_range:
    u_a = terzaghi_analytical(z_fine, t)
    U_a = 1.0 - np.trapz(u_a, z_fine) / (u0 * H)
    U_analytical_list.append(max(0, min(1, U_a)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_range, U_analytical_list, 'b-', linewidth=2.5, label='Analytical')
ax.plot(t_range, U_pinn, 'r--', linewidth=2, label='PINN')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Average Degree of Consolidation, U')
ax.set_title('Consolidation Progress  |  PINN vs Analytical')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Practice Exercises

Try these exercises to deepen your understanding:

```python
# Exercise 1: Double drainage
# Change boundary conditions to free drainage at both top and bottom.
# BC: u(0, t) = 0 AND u(H, t) = 0
# Update the loss function and compare with the double-drainage analytical solution.

# Exercise 2: Inverse problem
# Given measurement data, can the PINN estimate cv?
# Make cv a trainable parameter: self.cv = nn.Parameter(torch.tensor(1.0))

# Exercise 3: Loss weight sensitivity
# Try different weight combinations for pde, bc, ic, data terms.
# Which weights give the best accuracy?

# Exercise 4: Very sparse data
# Reduce n_data to 5 measurements. How does the PINN compare to data-only now?
# The physics constraint should help more when data is scarce.
```

---

## Key Takeaways

1. **PINNs embed governing equations** directly into the neural network loss function
2. **Automatic differentiation** computes PDE derivatives of the network output
3. **The loss has four components**: PDE residual, boundary conditions, initial conditions, and data
4. **PINNs outperform pure data-driven models** especially with sparse data
5. **The physics constraint** ensures physically plausible predictions even in data-scarce regions

---

## How This Connects

| ML Concept | Geotechnical Application |
|-----------|------------------------|
| PDE in loss | Terzaghi equation constrains consolidation predictions |
| Collocation points | Where the PDE is enforced across the soil domain |
| Boundary conditions | Drainage boundaries, impermeable layers |
| Physics + data | Combining sparse piezometer readings with theory |
| Inverse problems | Estimating soil parameters from field measurements |